In [1]:
import pandas as pd
import numpy as np
import re
import joblib

In [2]:
print("Loading model artifacts from 'models/' directory...")

iso_forest = joblib.load('models/isolation_forest_model.pkl')
kmeans = joblib.load('models/kmeans_cluster_model.pkl')
vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
pca = joblib.load('models/pca_transformer.pkl')
scaler = joblib.load('models/scaler.pkl')

print("All artifacts loaded successfully!")

Loading model artifacts from 'models/' directory...
All artifacts loaded successfully!


In [3]:
cluster_names = {0.0: 'Value/Quality Dissatisfaction',1.0: 'Rating Mismatch / Suspected Fake',2.0: 'Standard Product Return Risk'}

sentiment_map = {'Positive': 1.0,'Neutral': 0.5,'Negative': 0.0}

shrinkage_terms = r'\b(empty|missing|stolen|tampered|unsealed|swapped|used|fake|broken|tape)\b'

Mock Input

In [5]:
new_reviews_data = [{
        'Review_ID': 'REV_9901',
        'Review_Title': 'Box was completely empty!',
        'Review_Text': 'I opened the package and the seals were broken, tape was tampered with, and product stolen.',
        'Star_Rating': 1,
        'Quality_Sentiment': 'Negative',
        'Verified_Purchase': True,
        'Is_Repeat_Customer': False,
        'Return_Mentioned': True,
        'Complaint_Mentioned': True,
        'Is_Potentially_Fake': False,
        'Fake_Review_Score': 0.12
    },{
        'Review_ID': 'REV_9902',
        'Review_Title': 'Great purchase!',
        'Review_Text': 'Loved the product, fast shipping and good quality packaging.',
        'Star_Rating': 5,
        'Quality_Sentiment': 'Positive',
        'Verified_Purchase': True,
        'Is_Repeat_Customer': True,
        'Return_Mentioned': False,
        'Complaint_Mentioned': False,
        'Is_Potentially_Fake': False,
        'Fake_Review_Score': 0.02
    }]

In [6]:
input_df = pd.DataFrame(new_reviews_data)

In [7]:
input_df['Review_Title'] = input_df['Review_Title'].fillna('')
input_df['Full_Review'] = (input_df['Review_Title'] + " " + input_df['Review_Text']).str.lower()

In [8]:
input_df['Quality_Sentiment_Num'] = pd.to_numeric(input_df['Quality_Sentiment'].map(sentiment_map), errors='coerce').fillna(0.5)

In [9]:
normalized_rating = pd.to_numeric(input_df['Star_Rating'], errors='coerce') / 5.0
input_df['Rating_Quality_Mismatch'] = np.abs(normalized_rating - input_df['Quality_Sentiment_Num'])

In [10]:
input_df['Shrinkage_Keyword_Flag'] = input_df['Full_Review'].apply(lambda text: 1 if re.search(shrinkage_terms, str(text)) else 0)

In [11]:
tfidf_feats = vectorizer.transform(input_df['Full_Review'])
pca_feats = pca.transform(tfidf_feats.toarray())
input_df['Text_PCA1'] = pca_feats[:, 0]
input_df['Text_PCA2'] = pca_feats[:, 1]

In [13]:
X_new_unscaled = input_df.select_dtypes(include=[np.number, bool]).copy().astype(float)
X_new_scaled = scaler.transform(X_new_unscaled)

In [14]:
input_df['Anomaly_Flag'] = iso_forest.predict(X_new_scaled)
input_df['Is_Shrinkage_Anomaly'] = input_df['Anomaly_Flag'].apply(lambda x: 1 if x == -1 else 0)
input_df['Anomaly_Score'] = iso_forest.decision_function(X_new_scaled)

In [15]:
input_df['Shrinkage_Cluster'] = np.nan
anomaly_mask = input_df['Is_Shrinkage_Anomaly'] == 1

In [16]:
if anomaly_mask.any():
    anomaly_indices = input_df[anomaly_mask].index
    input_df.loc[anomaly_indices, 'Shrinkage_Cluster'] = kmeans.predict(X_new_scaled[anomaly_mask])

input_df['Shrinkage_Cluster_Label'] = input_df['Shrinkage_Cluster'].map(cluster_names)

In [17]:
output_cols = ['Review_ID', 'Star_Rating', 'Quality_Sentiment', 'Is_Shrinkage_Anomaly', 'Anomaly_Score', 'Shrinkage_Cluster_Label']

print("\n--- Inference Predictions ---")
print(input_df[output_cols])


--- Inference Predictions ---
  Review_ID  Star_Rating Quality_Sentiment  Is_Shrinkage_Anomaly  \
0  REV_9901            1          Negative                     0   
1  REV_9902            5          Positive                     1   

   Anomaly_Score        Shrinkage_Cluster_Label  
0       0.008494                            NaN  
1      -0.013984  Value/Quality Dissatisfaction  
